In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, f1_score

In [ ]:
loan_df = pd.read_csv('loan_data.csv')
print(loan_df.info())
print(loan_df.describe())
print(loan_df.isnull().sum())

In [ ]:
# 1. Data Cleaning & EDA
num_cols = loan_df.select_dtypes(include='number').columns
cat_cols = loan_df.select_dtypes(include='object').columns

In [ ]:
num_imp = SimpleImputer(strategy='mean')
loan_df[num_cols] = num_imp.fit_transform(loan_df[num_cols])
cat_imp = SimpleImputer(strategy='most_frequent')
loan_df[cat_cols] = cat_imp.fit_transform(loan_df[cat_cols])

In [ ]:
loan_df.isnull().sum()

In [ ]:
class_count = loan_df['Loan_Approved'].value_counts()
plt.pie(class_count, labels=['No','Yes'], autopct='%1.1f%%')
plt.title('Is Loan Approved or not')

In [ ]:
gender_count = loan_df['Gender'].value_counts()
ax = sns.barplot(gender_count)
ax.bar_label(ax.containers[0])

edu_cnt = loan_df['Education_Level'].value_counts()
ax = sns.barplot(edu_cnt)
ax.bar_label(ax.containers[1])

In [ ]:
sns.histplot(
    data = loan_df,
    x = 'Applicant_Income',
    bins = 20
)

In [ ]:
sns.histplot(
    data=loan_df,
    x='Coapplicant_Income',
    bins=20
)

In [ ]:
sns.histplot(
    data=loan_df,
    x='Credit_Score',
    hue='Loan_Approved',
    bins=20,
    multiple='dodge'
)

In [ ]:
sns.boxplot(
    data=loan_df,
    x='Loan_Approved',
    y='Applicant_Income'
)

In [ ]:
sns.boxplot(
    data=loan_df,
    x='Loan_Approved',
    y='Coapplicant_Income'
)

In [ ]:
fig, axes = plt.subplots(2,3)
sns.boxplot(ax=axes[0, 0], data=loan_df, x='Loan_Approved', y='Applicant_Income')
sns.boxplot(ax=axes[0,1], data=loan_df, x='Loan_Approved', y='Credit_Score')
sns.boxplot(ax=axes[0,2], data=loan_df, x='Loan_Approved', y='DTI_Ratio')
sns.boxplot(ax=axes[1,0], data=loan_df, x='Loan_Approved', y='Collateral_Value')
sns.boxplot(ax=axes[1,1], data=loan_df, x='Loan_Approved', y='Loan_Amount')
sns.boxplot(ax=axes[1,2], data=loan_df, x='Loan_Approved', y='Savings')
plt.tight_layout()

In [ ]:
loan_df.drop(columns='Applicant_ID', inplace=True)
loan_df.columns
loan_df.info()

In [ ]:
le = LabelEncoder()
loan_df['Education_Level'] = le.fit_transform(loan_df['Education_Level'])
loan_df['Loan_Approved'] = le.fit_transform(loan_df['Loan_Approved'])

cols = ['Employment_Status','Marital_Status','Loan_Purpose','Property_Area','Gender','Employer_Category']
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
encoded = ohe.fit_transform(loan_df[cols])
encoded
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(), index=loan_df.index)
encoded_df.head()
loan_df = pd.concat([loan_df.drop(columns=cols), encoded_df], axis=1)

In [ ]:
num_cols = loan_df.select_dtypes(include='number')
corr_matrix = num_cols.corr()

plt.figure(figsize=(15,8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm'
)

In [ ]:
corr_matrix['Loan_Approved'].sort_values(ascending=False)

In [ ]:
# 2. Feature Engineering

loan_df["DTI_Ratio_sq"] = loan_df['DTI_Ratio'] ** 2
loan_df['Credit_Score_sq'] = loan_df['Credit_Score'] ** 2

X = loan_df.drop(columns=['Loan_Approved', 'Credit_Score', 'DTI_Ratio'])
y = loan_df['Loan_Approved']

# 3. Model Training (Logistic, KNN, NB)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Logistic Regression Model

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print('--- Logistic Regression Model Results ---')
print(f'Precision : {precision_score(y_test, y_pred): .2}')
print(f'Recall : {recall_score(y_test, y_pred): .2}')
print(f'f1 score : {f1_score(y_test, y_pred): .2}')
print(f'CM: {confusion_matrix(y_test, y_pred)} ')
print(f'Accuracy : {accuracy_score(y_test, y_pred): .2}')

In [ ]:
# KNN Model

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print('--- KNN Model Results ---')
print(f'Precision : {precision_score(y_test, y_pred): .2}')
print(f'Recall : {recall_score(y_test, y_pred): .2}')
print(f'f1 score : {f1_score(y_test, y_pred): .2}')
print(f'CM : {confusion_matrix(y_test, y_pred)}')
print(f'Accuracy : {accuracy_score(y_test, y_pred): .2}')

In [ ]:
# GaussianNB Model

model = GaussianNB()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print('--- GaussianNB Model Results ---')
print(f'Precision : {precision_score(y_test, y_pred): .2}')
print(f'Recall : {recall_score(y_test, y_pred): .2}')
print(f'f1 score : {f1_score(y_test, y_pred): .2}')
print(f'CM : {confusion_matrix(y_test, y_pred)}')
print(f'Accuracy : {accuracy_score(y_test, y_pred): .2}')

In [ ]:
# --- Logistic Regression Model Results ---
# Precision :  0.79
# Recall :  0.8
# f1 score :  0.8
# CM: [[126  13]
#  [ 12  49]] 
# Accuracy :  0.88

# --- GaussianNB Model Results ---
# Precision :  0.78
# Recall :  0.77
# f1 score :  0.78
# CM : [[126  13]
#  [ 14  47]]
# Accuracy :  0.86

# --- KNN Model Results ---
# Precision :  0.62
# Recall :  0.51
# f1 score :  0.56
# CM : [[120  19]
#  [ 30  31]]
# Accuracy :  0.76

In [ ]:
# 4. Model Performance Comparison

data = {
    'Model': ['Logistic Regression', 'Gaussian NB', 'KNN'],
    'Precision': [0.79, 0.78, 0.62],
    'Recall': [0.80, 0.77, 0.51],
    'F1-Score': [0.8, 0.78, 0.56]
}

df_plot = pd.DataFrame(data)

df_melted = df_plot.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")
ax = sns.barplot(data=df_melted, x='Model', y='Score', hue='Metric', palette='viridis')

plt.title('Model Performance Comparison: CreditWise Loan System', fontsize=14, pad=20)
plt.ylim(0, 1.1) 
plt.ylabel('Score (0-1.0)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

for p in ax.patches:
    ax.annotate(format(p.get_height(), '.2f'), 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'center', 
                xytext = (0, 9), 
                textcoords = 'offset points',
                fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300)
plt.show()